<a href="https://colab.research.google.com/github/milesjustin05/milesjustin05.github.io/blob/master/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    AutoTokenizer,
    set_seed as hf_set_seed,
)
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from bayes_opt import BayesianOptimization
import random
import torch
import json
import os
from google.colab import userdata

In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
file_path = "Combined Data.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "suchintikasarkar/sentiment-analysis-for-mental-health",
  file_path,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
SEEDS = [42, 67, 69]

def set_seed(i):
    s = SEEDS[i]
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return s

In [ ]:
# Model initilization:

label2id={"Normal": 0, "Depression": 1, "Suicidal": 2, "Anxiety": 3, "Stress": 4, "Bipolar": 5, "Personality disorder": 6}
id2label={0: "Normal", 1: "Depression", 2: "Suicidal", 3: "Anxiety", 4: "Stress", 5: "Bipolar", 6: "Personality disorder"}
df["label"] = df["status"].map(label2id)

def model_init(model_name):
  return AutoModelForSequenceClassification.from_pretrained(
      model_name, num_labels=7, id2label=id2label, label2id=label2id)

In [ ]:
def tokenize_function(tok, examples):
    return tok(examples['statement'], truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall,
    }

In [ ]:
import gc, shutil

def training_iteration(model_name, tok, train_dataset, val_dataset, seed, trial_no,
                       lr, weight_decay, warmup_steps):
  hf_set_seed(seed)
  model = model_init(model_name)               # fresh weights every trial
  out_dir = f"runs/{model_name}/seed{seed}/n{len(train_dataset)}/trial{trial_no}"

  args = TrainingArguments(
      output_dir=out_dir,
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      learning_rate=lr,
      weight_decay=weight_decay,
      num_train_epochs=3,
      warmup_steps=int(warmup_steps),
      eval_strategy="epoch",
      save_strategy="no",
      logging_strategy="epoch",
      load_best_model_at_end=False,
      metric_for_best_model="f1_macro",
      greater_is_better=True,
      bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
      fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
      seed=seed,
      data_seed=seed,
      report_to="none",
  )

  trainer = Trainer(
      model=model, args=args,
      train_dataset=train_dataset, eval_dataset=val_dataset,
      processing_class=tok,
      data_collator=DataCollatorWithPadding(tokenizer=tok),
      compute_metrics=compute_metrics,
  )
  trainer.train()

  raw_history = pd.DataFrame(trainer.state.log_history)

  history_df = (
    raw_history.dropna(subset=["epoch"])
    .groupby("epoch", as_index=False)
    .first()
    .drop(
        columns=[
            "train_runtime",
            "train_samples_per_second",
            "train_steps_per_second",
            "total_flos",
            "train_loss",
        ],
        errors="ignore",
    )
    .rename(columns={"loss": "train_epoch_loss"})
)

  history_df["model"] = model_name
  history_df["trial"] = trial_no
  history_df["seed"] = seed
  history_df["n_train"] = len(train_dataset)
  history_df["lr"], history_df["weight_decay"], history_df["warmup_steps"] = lr, weight_decay, int(warmup_steps)

    # 1. Compute THIS trial's best F1 BEFORE concatenating with previous trials!
  f1 = float(history_df["eval_f1_macro"].max())

  # 2. Append this trial's history to training_history.json
  path = "training_history.json"
  if os.path.exists(path) and os.path.getsize(path) > 0:
    with open(path, "r", encoding="utf-8") as f:       # was opening file_path (the CSV name)
      history_df = pd.concat([pd.DataFrame(json.load(f)), history_df], ignore_index=True)
  history_df.to_json(path, orient="records", indent=4)

  del trainer, model
  gc.collect(); torch.cuda.empty_cache()
  shutil.rmtree(out_dir, ignore_errors=True)           # don't fill the disk with checkpoints
  return f1

In [ ]:
def optimize(model_name, tok, train_dataset, val_dataset, seed):
  steps = (len(train_dataset) / 16) * 3

  pbounds = {
      "log_lr":       (np.log10(1e-5), np.log10(5e-5)),   # same range, log scale
      "weight_decay": (0.01, 0.1),
      "warmup_ratio": (0.05, 0.10),                        # same 5–10% as before, now fraction-independent
  }

  trial = {"n": 0}

  def objective(log_lr, weight_decay, warmup_ratio):
    trial["n"] += 1
    lr = 10 ** log_lr
    warmup_steps = int(warmup_ratio * steps)
    return training_iteration(model_name, tok, train_dataset, val_dataset, seed,
                              trial["n"], lr, weight_decay, warmup_steps)

  optimizer = BayesianOptimization(f=objective, pbounds=pbounds, random_state=seed, verbose=2)
  optimizer.set_gp_params(
      kernel=Matern(nu=2.5, length_scale=np.ones(3)) + WhiteKernel(noise_level=1e-3)
  )

  optimizer.maximize(init_points=3, n_iter=7)

  print("Best parameters:", optimizer.max["params"])
  print("Best val F1:", optimizer.max["target"])
  return optimizer

In [ ]:
def save_seed_result(optimizer, model_name, seed, frac, n_train):
  steps = (n_train / 16) * 3
  record = {
      "model": model_name,
      "seed": seed,
      "train_fraction": frac,
      "n_train": n_train,
      "best_val_f1-macro": optimizer.max["target"],
      "learning rate": 10 ** optimizer.max["params"]["log_lr"],
      "weight decay": optimizer.max["params"]["weight_decay"],
      "warmup ratio": optimizer.max["params"]["warmup_ratio"],
      "warmup steps": int(optimizer.max["params"]["warmup_ratio"] * steps),
  }

  path = "model.json"
  data = []

  if os.path.exists(path) and os.path.getsize(path) > 0:
    with open(path, "r", encoding="utf-8") as f:
      data = json.load(f)
  data.append(record)
  with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)


In [ ]:
from functools import partial

models = ['bert-base-uncased']
FRACS = [0.03125, 0.0625, 0.125, 0.25, 0.50, 1.0]   # fraction of the training pool; edit to your schedule

df = df.dropna(subset=["statement", "label"]).drop_duplicates(subset="statement").reset_index(drop=True)
df["statement"] = df["statement"].astype(str)
df["label"] = df["label"].astype(int)

def nested_subsets(train_df, fracs, seed):
  """Stratified, nested subsets: subset(f1) is contained in subset(f2) whenever f1 < f2."""
  shuffled = train_df.sample(frac=1.0, random_state=seed).copy()
  shuffled["_rank"] = shuffled.groupby("label").cumcount()
  class_sizes = shuffled["label"].value_counts()
  out = {}
  for frac in fracs:
    keep_n = np.ceil(frac * class_sizes).astype(int)          # per-class quota
    mask = shuffled["_rank"] < shuffled["label"].map(keep_n)
    out[frac] = shuffled[mask].drop(columns="_rank")
  return out

for model_name in models:
  tok = AutoTokenizer.from_pretrained(model_name)
  tok_fn = partial(tokenize_function, tok)

  for j in range(3):
    seed = set_seed(j)

    train_val_df, test_df = train_test_split(
        df, test_size=0.2, stratify=df["label"], random_state=seed)
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.125, stratify=train_val_df["label"], random_state=seed)

    val_dataset = Dataset.from_pandas(val_df[["statement", "label"]], preserve_index=False).map(tok_fn, batched=True)

    subsets = nested_subsets(train_df, FRACS, seed)
    for frac in FRACS:
      sub_df = subsets[frac]

      train_dataset = Dataset.from_pandas(sub_df[["statement", "label"]], preserve_index=False).map(tok_fn, batched=True)

      optimizer = optimize(model_name, tok, train_dataset, val_dataset, seed)
      save_seed_result(optimizer, model_name, seed, frac, len(sub_df))